![logo](https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/docs/source/_figures/Helmholtz-AI.png?raw=true)

# Welcome to the fine-tuning tutorial!

Here, we will learn how to use knowledge that is already in the model's weights to fine-tune it for a task different than it's original task. We will tackle domain-adaptation and finetuning in one shot! Specifically, we will focus on two different tasks:
1. Fine-tune BERT on a sentiment analisys task.
2. Fine-tune VIT on an new calssification task.


Feel free to run the notebook and experiment with it. In addition, we have left a few sections empty ot give you the chance to pt into practice what you have leared today.

Let's start with  installing the required packages:

## Getting Started

### Setup Colab environment

If you installed the packages and requirements on your machine, you can skip this section and start from the import section.
Otherwise, you can follow and execute the tutorial on your browser. To start working on the notebook, click on the following button. This will open this page in the Colab environment, and you will be able to execute the code on your own.

<a href="https://colab.research.google.com/github/HelmholtzAI-Consultants-Munich/llm-transformers-course/blob/main/Transformer_finetuning_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Now that you opened the notebook in Google Colab, follow the next step:

1. Run this cell to connect your Google Drive to Colab and install packages
2. Allow this notebook to access your Google Drive files. Click on 'Yes', and select your account.
3. "Google Drive for desktop wants to access your Google Account". Click on 'Allow'.
   
At this point, a folder has been created in your Drive, and you can navigate it through the lefthand panel in Colab. You might also have received an email that informs you about the access on your Google Drive.

In [ ]:
# Mount drive folder to dbe abale to download repo
# from google.colab import drive
# drive.mount('/content/drive')

# Switch to correct folder'
# %cd /content/drive/MyDrive

In [ ]:
# Don't run this cell if you already cloned the repo 
# %rm -r llm-trasformers-course
# !git clone --branch main https://github.com/HelmholtzAI-Consultants-Munich/llm-transformers-course.git

In [ ]:
# Install al required dependencies and package versions
# %cd llm-transformers-course
# !pip install -r requirements.txt --> CREATE requirements txt and remove next cell

In [ ]:
!pip install torch evaluate transformers matplotlib scikit-learn seaborn pillow torchinfo

Before we start, make sure you're using the GPU or TPU execution envionment. The code below should output `True`:

In [ ]:
import torch
print(torch.cuda.is_available())

## Fine-Tuning BERT for sentiment analysis

Let us begin with downloading the pretrained model. Here, we will use the `google-bert/bert-base-uncased` model, an encoder-only model that is pre-trained on the masked token prediction task.
More in details, the pre-training dataset consists in a series of sentences, where a percentage of the tokens have been masked (or hidden), and the task of the model is to predict the original token. This forces the model to learn contextual relationships between words.

<img src="https://tinyurl.com/54zye66r" alt="Masked token predition" class="center" width="500"/>

If you're interested in learning more about the model itself, refer to the model card here: [Model card](https://huggingface.co/google-bert/bert-base-uncased)

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM
from transformers import ViTImageProcessor, ViTForImageClassification
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.decomposition import PCA
from evaluate import load
from sklearn.metrics import confusion_matrix
from PIL import Image
from datasets import load_dataset

accuracy = load("accuracy")
import torchinfo
import torch
import requests
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch.nn.functional as F


model = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

Let's see if we can use the model for it's original task, which is masked prediction.

We first need to use the tokenizer to convert the human-readable text to the tokens that are understandable for the model. Then, we call the model directly. Finally, we have to revert back to human-readable text, so we once again use the tokenizer. It's important to use the correct tokenizer in order to feed to the model meaningful embeddings.

Most models in the `transformers` framework always return a sequence (batch) of outputs, even if we just input one sentence to process. That's why we have to iterate over the outputs.

In [ ]:
tokenized_inputs = tokenizer("Berlin is the [MASK] of Germany.", return_tensors="pt")
predicted = model(**tokenized_inputs)

# Get the predicted logits for the token that was masked
mask_token_index = torch.where(tokenized_inputs["input_ids"] == tokenizer.mask_token_id)[1]
logits = predicted.logits[0, mask_token_index, :]

# get the prediction with the largest logit
predicted_token = torch.argmax(logits, dim=1)

# Decode the tokens to words
for token in predicted_token:
    print("Berlin is the [MASK] of Germany.".replace("[MASK]", tokenizer.decode([token])))

The bert tokenizer not only tokenizes text, it also appends some control tokens:

In [ ]:
tokenizer.convert_ids_to_tokens(tokenized_inputs["input_ids"][0])

The bert family of models always outputs a special `[CLS]` token as the first token in the output. This token is normally not visible after decoding, but it can be used directly for classification, as by definition it is supposed to summarize the information from the whole sequence.

If you would like to understand how the BERT model learns what the [CLS] token should look like, take a look at the [BERT-NSP paper](https://arxiv.org/abs/2109.03564).

After the whole sentence is tokenized, the tokenizer also appends the `[SEP]` token, which denotes the end of sequence (or sequence separation, if multiple sentences are encoded). We can also see the token id for each of the tokens:

In [ ]:
for id, token in zip(tokenized_inputs["input_ids"][0], tokenizer.convert_ids_to_tokens(tokenized_inputs["input_ids"][0])):
  print(f"{token}\t{id}")

We mentioned before that the `[CLS]` token should contain the summarized meaning of the whole sentence.

For text in the encoded latent space, we can expect similar pieces of text to lie close to each other. Let's see if this is the case! We can encode our sentences and see where they lie after projecting to a 2D space, only by looking at the `[CLS]` token.

The `AutoModelForMaskedLM` class that we used before contains a classification head that outputs logits that help to decide which token to choose, but if we want to obtain representations of the sentences, we should use just a simple `AutoModel`.

In [ ]:
model = AutoModel.from_pretrained("google-bert/bert-base-uncased")

In [ ]:
sentences = [
    "I like cats",
    "My kitchen has a sink",
    "I like dogs",
    "I own a fridge",
    "I hate Metallica",
    "My mum hates Bon Jovi",
    "I have a new coffee machine"
]

encoder_inputs = [
    tokenizer(sentence, return_tensors="pt")
    for sentence in sentences
]

encoder_outputs = [
    model(**input).last_hidden_state for input in encoder_inputs
]

We now obtain a numerical, vectorized representation of our sentences:

In [ ]:
for sentence, encoding in zip(sentences[:2], encoder_outputs):
  print(sentence, "\nencoding:", encoding.detach().numpy(), encoding.detach().shape)

Let's now apply PCA to the encodings to project them in 2D and see if there's any relationship between them! We only look at the `[CLS]` token, which always comes first in the sequence.

In [ ]:
encoder_outputs_cls = [
    encoding.detach().numpy()[0,0,:]
    for encoding in encoder_outputs
]

encoded_outputs_2d = PCA(n_components=2).fit_transform(encoder_outputs_cls)

plt.scatter(encoded_outputs_2d[:, 0], encoded_outputs_2d[:, 1])

for i, sentence in enumerate(sentences):
    plt.annotate(sentence, (encoded_outputs_2d[i, 0]+.1, encoded_outputs_2d[i, 1]+.1))

plt.xlabel("PC1")
plt.ylabel("PC2")

plt.show()

Our latent space seems to make sense! Senteces related to liking a pet are very close together. Dislike is also expressed by points lying close in the latent space, and kitchen appliances form a third cluster.

This is a very important result as it assures us that our model has some basic understanding of the English language.

We can now proceed with finetuning!

**Finetuning**

We begin with loading our dataset and seeing what kind of data we're dealing with.

The `imdb` dataset is a simple classification dataset. The inputs are movie reviews, and the expected outputs are whether the review is generally positive or negative. The classes (positive/negative) are also balanced.

We downsize the training, validation and test sets to speed-up the fine-tuning.

📋 Optional: when you're done with the notebook, try to retrain with the full dataset. Does it change the accuracy much?

In [ ]:
from datasets import load_dataset
imdb = load_dataset("imdb").shuffle(seed=42)

train_size = 8096
val_size = 1024
test_size = len(imdb["test"]) // 10

imdb["val"] = imdb["train"].select(range(train_size, train_size+val_size))
imdb["train"] = imdb["train"].select(range(train_size))

imdb["test"] = imdb["test"].select(range(test_size))


Let's have a look how the dataset is structured. For each data instance we have a `"text"` and a `"label"` entry, where the first contains the actual review and the second one `1` or `0`, depending on the assigned class.



In [ ]:
for entry in range(5):
  print("\n", imdb["train"]["text"][entry])
  label = imdb["train"]["label"][entry]
  print("\t", "Positive" if label else "Negative", f"({label})")


print("Proportion of positive reviews in the test set:", sum(imdb["test"]["label"])/len(imdb["test"]))

It's a straightforward text-classification task with balanced classes, we want to predict whether a review is positive or negative just by looking at the text.

Let us build a classification model. Please take a look again at the `[CLS]` token. In theory, it should summarize the information from the entire sentence, which is exactly want we want here. We would like to predict whether the review is negative or positive, so we don't need to look at specific words, we're interested in the entire sentence.


We can use this to build our model. We just need to create our own classification head, which in our case is a simple Feedforward Neural Network that receives in input the `[CLS]` embedding and returns the class probabilities.

In [ ]:
from transformers import AutoModel
import torch
import torch.nn as nn
import torch.nn.functional as F

class SentimentClassificationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained("bert-base-uncased")

        self.clf_head = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 1)
        )

    def forward(self, input_ids, attention_mask, token_type_ids, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        # Take the [CLS] token embedding (first token)
        cls_embedding = outputs.last_hidden_state[:, 0, :]

        logits = self.clf_head(cls_embedding)

        if labels is not None:
            # Binary cross-entropy with logits
            loss = F.binary_cross_entropy_with_logits(logits.squeeze(), labels.float())
            return loss, torch.sigmoid(logits)

        return torch.sigmoid(logits)


In [ ]:
sentiment_clf_model = SentimentClassificationModel()

sample_input = tokenizer("I HATED this movie!!!", return_tensors="pt")
# the tokenizer outputs a dictionary with keys {input_ids, attention_mask}.
# we can use dictionary unpacking when calling the model
sentiment_clf_model(**sample_input)

Our model already outputs a simple score. Currently, it's far from perfect, for such an obvious input, the prediction should be very close to `0.0`. We can now proceed with fine-tuning.



Since our representations in the latent space already make sense, we don't need to fine-tune the whole model. We can freeze the weights of the encoder and just train the classification head.

In [ ]:
for parameter in sentiment_clf_model.encoder.parameters():
  parameter.requires_grad = False

This greatly reduces the number of parameters that we actually need to affect during training:

In [ ]:
trainable_params = sum(p.numel() for p in sentiment_clf_model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in sentiment_clf_model.parameters())
print(f"% of trainable parameters: {trainable_params/all_params*100:.2f}% \ntrainable param./ total param. = {trainable_params} /{all_params} total param.")

Now, we need to prepare the dataset for training. The transformers trainer expects the input to already be tokenized. We process the whole dataset at once and store it for later.

We also define how we compute accuracy.

In [ ]:
accuracy = load("accuracy")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.round(predictions).squeeze()
    return accuracy.compute(predictions=predictions, references=labels)

tokenized_imdb = imdb.map(tokenize_function, batched=True)
tokenized_imdb = tokenized_imdb.remove_columns(["text"])
tokenized_imdb = tokenized_imdb.rename_column("label", "labels")

Now, we define the parameters of the training. We will train for 5 epochs initially.

In [ ]:

training_args = TrainingArguments(
    output_dir='.',
    learning_rate=1e-4,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    fp16=True,
    logging_steps=1,
)

trainer = Trainer(
    model=sentiment_clf_model,
    args=training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["val"],
    compute_metrics=compute_metrics,
)

Now, we move our model to the GPU and train it.

In [ ]:
sentiment_clf_model.to("cuda")

trainer.train()

After 5 epochs you might notice that the accuracy is acceptable, but not perfect. This is because we only actually trained a tiny fraction of the model's weights, completely relying on the encoder to be able to extract all the important features.

We will now try training the entire model to see if the performance improves. We can expect that to happen, since BERT is a general purpose model, and the embeddings produced by it might not be optimal for the sentiment analysis task.

📋 Exercise:
1. Modify the code we used above to freeze the model's weights to unfreeze the whole model
2. Define the training arguments
3. Train the model

<details>
  <summary>💡 Tipps</summary>
   We need to  lower the batch size now to 48, as the parameter count increases about 100-fold and. You can also reduce the number of epochs to 2 to make the training faster
</details>



In [ ]:
# Exercise...

# YOUR CODE

📋 Optional: Once you're done with the notebook, try to train the model until convergence.

📋 Optional: With the above, try inclduing an early stopping callback that makes use of the validation set.

Our model is now trained and the validation results look more promising! Let's see the final test set accuracy and some misclassified samples.

In [ ]:
test_results = trainer.evaluate(tokenized_imdb["test"])

print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")

predictions = trainer.predict(tokenized_imdb["test"]).predictions
predicted_labels = np.round(predictions).squeeze()

actual_labels = tokenized_imdb["test"]["labels"]

correct_indices = np.where(predicted_labels == actual_labels)[0].tolist()
incorrect_indices = np.where(predicted_labels != actual_labels)[0].tolist()

print("\nCorrectly Classified Examples:")
for i in correct_indices[:5]:
    print(f"Review: {imdb['test']['text'][i]}")
    print(f"Actual: {'Positive' if actual_labels[i] else 'Negative'}, Predicted: {'Positive' if predicted_labels[i] else 'Negative'}\n")

print("\nIncorrectly Classified Examples:")
for i in incorrect_indices[:5]:
    print(f"Review: {imdb['test']['text'][i]}")
    print(f"Actual: {'Positive' if actual_labels[i] else 'Negative'}, Predicted: {'Positive' if predicted_labels[i] else 'Negative'}\n")

We can also take a look at the confusion matrix. In this case, we could try to see if the model is more likely to confuse true negative labels, but generally performs quite well.

In [ ]:

cm = confusion_matrix(actual_labels, predicted_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

📋 Exercise: run the trained model on a review you wrote for a movie you saw. Did the model get your sentiment right? Try modifying your review to confuse it.

In [ ]:
# Exercise...

# YOUR CODE

### VIT for image classification

Transformers can work not only with text. Here, we will explore the ViT model. ViT stands for *Vision Transformer*, and as the name implies is used to analyze images. Refer to the [model card](https://huggingface.co/google/vit-base-patch16-224) of the ViT model we are using here, if you're interested in more details.

<img src="https://tinyurl.com/5n6d4pj4" alt="VIT" class="center" width="500"/>


As before, we begin by loading the model and it's processor:

In [ ]:

processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

The ViT model we use here is already pretrained on the ImageNet dataset, which contains images from 1000 diverse different classes, like pet breeds, means of transportation or everyday objects. Let's see an example image:

In [ ]:
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)
image

We can obtain the model's prediction for the image by using the processor on it and subsequently apply the model to the processed inputs.

In [ ]:
inputs = processor(images=image, return_tensors="pt")
outputs = model(**inputs)
logits = outputs.logits
# model predicts one of the 1000 ImageNet classes
# we select the class with the highgest probability
predicted_class_idx = logits.argmax(-1).item()
print("Predicted class:", model.config.id2label[predicted_class_idx])

Let us try to fine-tune our model to a different task. In this case, we are interested in predicting whether a bean leaf is healthy or diseased. Let's take a look:

In [ ]:
beans = load_dataset("AI-Lab-Makerere/beans").shuffle(seed=42)
beans

In [ ]:
label_mapping = beans["train"].features["labels"].int2str

for idx, d in list(enumerate(beans["train"]))[:9]:
  image = d["image"]
  label = d["labels"]

  plt.subplot(3, 3, idx+1)
  plt.imshow(image)
  plt.axis("off")
  plt.title(label_mapping(label))
plt.show()


Our new dataset contains photos of bean leaves and a classification into one of the three classes: healthy, spotty and rusty. The task is to predict the target class.

Let's see if our model can do it right away:

In [ ]:
prediction = model(**processor(beans["train"][0]["image"], return_tensors="pt"))
prediction[0].shape

Oops! We seem to have a problem. We have just three classes (healthy, rusty, spotty), but the model outputs a thousand, because it was trained on a different dataset.
We can take a look at the last layer that is the cause of this issue:

In [ ]:
torchinfo.summary(model, depth=2, row_settings=["var_names", "depth"])

In [ ]:
print(model.classifier)

Thankfully, we can reload the model in such a way that it predicts only 3 classes, while preserving everything it has learned from the ImageNet dataset:

In [ ]:
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224', num_labels=3, ignore_mismatched_sizes=True)

As we can see above, the model loads all the available weights, but skips the classifier head's weights as the output size is now different:

In [ ]:
print(model.classifier)

Since the model is designed for image classification, we don't need a wrapper class like before, we can proceed with finetuning straight away. As before, we need to preprocess the inputs, and define the training arguments:

In [ ]:
def process_function_beans(examples):
    return processor(examples["image"])

tokenized_beans = beans.map(process_function_beans, batched=True)

In [ ]:
training_args = TrainingArguments(
    output_dir='.',
    learning_rate=1e-4,
    per_device_train_batch_size=56,
    per_device_eval_batch_size=56,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    fp16=True,
    logging_steps=1,
)

def compute_metrics_vit(eval_pred):
    predictions, labels = eval_pred
    # The model outputs logits, which are floating-point numbers.
    # We need to convert them to class predictions (integers) for accuracy calculation.
    predicted_labels = predictions.argmax(axis=-1)
    return accuracy.compute(predictions=predicted_labels, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_beans["train"],
    eval_dataset=tokenized_beans["validation"],
    compute_metrics=compute_metrics_vit,
)

And we are ready to train the model:

In [ ]:
model.to('cuda')

trainer.train()

In [ ]:
test_results = trainer.evaluate(tokenized_beans["test"])

print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")

The test accuracy is not bad even though we only trained for one epoch. Let's now see the predictions of our new model:

In [ ]:
predictions = trainer.predict(tokenized_beans["test"])
predicted_logits = predictions.predictions

In [ ]:
predicted_labels = predicted_logits.argmax(axis=-1)
actual_labels = tokenized_beans["test"]["labels"]

correct_indices = np.where(predicted_labels == actual_labels)[0].tolist()
np.random.shuffle(correct_indices)
incorrect_indices = np.where(predicted_labels != actual_labels)[0].tolist()
np.random.shuffle(incorrect_indices)

In [ ]:
label_mapping = beans["train"].features["labels"].int2str
unique_labels = sorted(list(set(beans["train"]["labels"])))
print("Label mapping:", {i: label_mapping(i) for i in unique_labels})

num_display = 5
fig, axes = plt.subplots(2, num_display, figsize=(20, 8))
fig.suptitle('Correctly and Incorrectly Classified Images', fontsize=16)

for i in range(num_display):
    idx = correct_indices[i]
    image = beans["test"]["image"][idx]
    actual = label_mapping(int(actual_labels[idx]))
    predicted = label_mapping(int(predicted_labels[idx]))

    axes[0, i].imshow(image)
    axes[0, i].set_title(f"Actual: {actual}\nPredicted: {predicted}")
    axes[0, i].axis('off')

for i in range(num_display):
    idx = incorrect_indices[i]
    image = beans["test"]["image"][idx]
    actual = label_mapping(int(actual_labels[idx]))
    predicted = label_mapping(int(predicted_labels[idx]))

    axes[1, i].imshow(image)
    axes[1, i].set_title(f"Actual: {actual}\nPredicted: {predicted}", color='red')
    axes[1, i].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

As before, we can also take a look at the confusion matrix:

In [ ]:
cm = confusion_matrix(actual_labels, predicted_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Healthy', 'Bean_rust', 'Angular_leaf_spot'], yticklabels=['Healthy', 'Bean_rust', 'Angular_leaf_spot'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

📋 Exercise:

Go back to the code, what can we do to increase the model accuracy?

<details>
  <summary>💡 Tipps </summary>

  1. Try changing the training hyperparameters.

  2. There are different versions of VIT, try using a larger model, does this help?
</details>
